# Splitting Annotation Work Among Annotators

We have annotations that were previously completed by Yaseen, Tan, and Daniel, plus new annotations to complete. We want to add one more annotator to the previously-completed annotations (Kumar or Parsa), and split the new annotations among the five annotators.

In [ ]:
import ast
import json
import random
from pathlib import Path
from typing import cast

import pandas as pd

In [ ]:
CSV_BASE = Path('').resolve().parent / 'csv'

`blackboard.csv` is the previously-completed annotations from Yaseen, Tan, and Daniel.

In [ ]:
df = pd.read_csv(CSV_BASE / 'blackboard.csv')
df.head()

Some of the instances contain multiple issue links.

In [ ]:
mask = (
    df['issue_link'].str.contains(' ')
    | df['issue_link'].str.contains('\n')
    | df['issue_link'].str.contains(',')
)
df_filtered = df[mask]
df_filtered.head()

This is the complete set of issue categories that we have used so for.

In [ ]:
print(df['issue_category'].unique().tolist())
[
    '[1.1 Incomplete Data Processing]',
    '[2.1 Incorrect Data Processing]',
    '[1.3 Error Handling]',
    '[1.2.3 Missing Null Check]',
    '[2.2 Incorrect Input Validation]',
    '[4 Perfective Maintenance]',
    '2.4 Incorrect output',
    '2.5 Incorrect configuration processing',
    '2.1 Incorrect data processing',
    '1.4 Incomplete configuration processing',
    '1.1 Incomplete data processing',
    '[2.2.2 Incorrect Handling of Special Characters]',
    '[2.4.1 Incorrect Output Message]',
    '[1.2.3 Missing null check]',
    '[2.1 Incorrect data processing]',
    '[1.2 Incomplete input validation]',
    '[2.4 Incorrect output]',
    '[1.4 Incomplete Configuration Processing]',
    '[1.1 Incomplete data processing]',
    '[2.5 Incorrect configuration processing]',
    '[2.1.2 Incorrect initialization]',
    '[4 Perfective maintenance]',
    '[2.7 Performance]',
    '[1.4 Incomplete configuration processing]',
    '2.2 Incorrect input validation',
    '[2.1.2 Incorrect Initialization]',
    '[3.3.1 Missing Initialization]',
    '[1.2 Missing Input Validation]',
    '[1.5 Incomplete Output Message]',
    '[1.2.1 Missing Type Check]',
    '4 Perfective maintenance',
    '2.7 Performance',
    '2.2.2 Incorrect handling of special characters',
    '1.2.5 Missing handling of special characters',
    '2.6 Incorrect handling of regex expressions',
    '1 Incomplete feature implementation',
    '2 Incorrect feature implementation',
    '1.3 Error handling',
    '1.2.3 Missing null check',
    '4 Perfective Maintenance',
]

In [ ]:
df['issue_category'] = df['issue_category'].str.strip().str.strip('[]')

This is the complete set of the first image categories that we have used to far.

In [ ]:
print(df['image_category_1'].unique().tolist())
[
    'Web Interface (UI/UX Element), Code Snippet Screenshot',
    'Code Snippet Screenshot, Error Message',
    'Map/Geospatial Visualization, Error Message',
    'Code Snippet Screenshot, Map/Geospatial Visualization',
    'Code Snippet Screenshot, Data Visualization',
    'Map/Geospatial Visualization, Web Interface (UI/UX Element)',
    'Code Snippet Screenshot, Diagram',
    'Diagram, Error Message',
    'Diagram, Web Interface (UI/UX Element), Error Message',
    'Diagram, Web Interface (UI/UX Element)',
    'Code Snippet Screenshot, Web Interface (UI/UX Element)',
    'Code Snippet Screenshot, Miscellaneous',
    'Error Message, Web Interface (UI/UX Element)',
    'Web Interface (UI/UX Element), Diagram',
    'Web Interface (UI/UX Element), Error Message',
    'Error Message, Diagram',
    'Code Snippet Screenshot, Map/Geospatial Visualization, Web Interface '
    '(UI/UX Element)',
    'Miscellaneous, Web Interface (UI/UX Element)',
    'Diagram, Web Interface (UI/UX Element), Code Snippet Screenshot, Error '
    'Message',
    'Diagram, Web Interface (UI/UX Element), Code Snippet Screenshot',
]

In [ ]:
df['image_category_1'] = df['image_category_1'].str.split(', ')
df = df.explode('image_category_1')

This is the complete set of the second image categories that we have used to far.

In [ ]:
print(df['image_category_2'].unique().tolist())
[
    'Dialog Box, Desired Output',
    'Code, Program Input',
    'Code, Run Time Error',
    'Steps and Processes, Algorithm/Concept Description',
    'Code, Program Output',
    'Code, Program Input, Desired Output',
    None,
    'Desired Output, Program Output',
    'Program Output, Desired Output',
    'Code, Program Input, Program Output',
    'Steps and Processes, Program Output',
    'Program Output, Run Time Error',
    'Program Output, Steps and Processes',
    'Code, Program Output, Menus and Preference',
    'Algorithm/Concept Description, Dialog Box',
    'Run Time Error, Algorithm/Concept Description',
    'Algorithm/Concept Description, Program Output',
    'Algorithm/Concept Description, Menus and Preference',
    'Algorithm/Concept Description, Run Time Error',
    'Steps and Processes, Code, Desired Output',
    'Code, Desired Output',
    'Code, Menus and Preference, Desired Output',
    'Program Input, Code',
    'Code, Menus and Preference, Program Output',
    'Code, Steps and Processes',
    'Code, Desired Output, Program Output',
    'Run Time Error, Program Output',
    'Program Output, Menus and Preference',
    'Run Time Error, Steps and Processes, Menus and Preference, Program '
    'Output',
    'Menus and Preference, Program Output, Steps and Processes',
    'Menus and Preference, Program Output, Steps and Processes, Run Time '
    'Error',
]

In [ ]:
df['image_category_2'] = df['image_category_2'].str.split(', ')
df = df.explode('image_category_2')

The image links (`image_assets`) is inconsistent: sometimes a list of strings, sometimes a string. The `parse_list` function parsed out the URLs.

In [ ]:
def parse_list(val: str) -> str | list[str]:
    if pd.isna(val):
        return val
    val = val.strip()
    try:
        result = ast.literal_eval(val)
        return (
            cast(list[str], result) if isinstance(result, list) else [result]
        )
    except (ValueError, SyntaxError):
        # Handles [https://...] style (unquoted)
        if val.startswith('[') and val.endswith(']'):
            return [u.strip() for u in val[1:-1].split(',')]
        return [val]


df['image_assets'] = df['image_assets'].apply(parse_list)
df = df.explode('image_assets')

In [ ]:
df.head()

`df_unresolved` is the subset of unresolved instances that matches the distribution of the resolved instances.

In [ ]:
df_unresolved = pd.read_csv(CSV_BASE / 'unresolved-subset.csv')

In [ ]:
print(df.columns)
print(df_unresolved.columns)

Now we melt the first DataFrame so that the `issue_category`, `image_category_1`, and `image_category_2` columns are converted into `key` and `value` columns, where the `key` is the original column name and the `value` is the value of the row for that column. This allows us to treat individual annotations as "events", simplifying the logic to save annotations. We can then pivot the events back into a table later.

In [ ]:
df_melted = df.melt(
    id_vars=[
        'name',
        'instance_id',
        'issue_link',
        'problem_statement',
        'image_assets',
    ],
    var_name='key',
    value_name='value',
)
df_melted.head()
df_melted.to_csv(CSV_BASE / 'events.csv', index=False)

Now we combine the previous annotations with the new unresolved-subset.

In [ ]:
df_combined = pd.concat(
    [df_melted, df_unresolved.reindex(columns=df_melted.columns)],
    ignore_index=True,
)
df_combined.head()

Again we ensure that the `image_assets` column is normalized.

In [ ]:
def normalize_image_assets(val: str) -> list[str] | None:
    if pd.isna(val):
        return None
    val = str(val).strip()
    try:
        parsed = json.loads(val)
        urls = [url for urls in parsed.values() for url in urls]
    except (json.JSONDecodeError, AttributeError):
        urls = [val]

    return urls


df_combined['image_assets'] = df_combined['image_assets'].apply(
    normalize_image_assets
)
df_combined = df_combined.explode('image_assets', ignore_index=True)
df_combined.head()

The logic to split the annotation work is as follows:

1. For all previously annotated rows, give half to Kumar and half to Parsa, so that each is reviewed one more time.
2. For new rows, create two pairings:
   - Pairing A: duplicate the row, give one to Kumar, one to Yaseen.
   - Pairing B: duplicate the row twice, give one to Parsa and the other to either Tan or Daniel.

In [ ]:
notna_idx = df_combined[df_combined['name'].notna()].index.tolist()
na_idx = df_combined[df_combined['name'].isna()].index.tolist()

# Shuffle and split exactly 50/50
random.shuffle(notna_idx)
notna_kumar = set(notna_idx[: len(notna_idx) // 2])
notna_parsa = set(notna_idx[len(notna_idx) // 2 :])

random.shuffle(na_idx)
na_pairing_a = set(na_idx[: len(na_idx) // 2])
na_pairing_b = set(na_idx[len(na_idx) // 2 :])

# For pairing B, split tan/daniel exactly 50/50
na_pairing_b_list = list(na_pairing_b)
random.shuffle(na_pairing_b_list)
na_b_tan = set(na_pairing_b_list[: len(na_pairing_b_list) // 2])
na_b_daniel = set(na_pairing_b_list[len(na_pairing_b_list) // 2 :])

full_cols = list(df_combined.columns)
keep_cols = [c for c in full_cols if c not in ['key', 'value']]
rows: list[pd.Series] = []

df_old_rows = df_combined[df_combined['name'].notna()][full_cols]

for idx, row in df_combined.iterrows():
    if pd.notna(row['name']):
        r = row[keep_cols].copy()
        r['name'] = 'kumar' if idx in notna_kumar else 'parsa'
        r['_from_na'] = False
        rows.append(r)
    else:
        base = row[keep_cols].copy()
        r1, r2 = base.copy(), base.copy()
        if idx in na_pairing_a:
            r1['name'] = 'kumar'
            r2['name'] = 'yaseen'
        else:
            r1['name'] = 'parsa'
            r2['name'] = 'tan' if idx in na_b_tan else 'daniel'
        r1['_from_na'] = True
        r2['_from_na'] = True
        rows.extend([r1, r2])

df_new_rows = pd.DataFrame(rows, columns=keep_cols + ['_from_na'])
df_final = pd.concat([df_old_rows, df_new_rows], ignore_index=True)

In [ ]:
df_final.head()

Next, we perform a series of assertions to ensure that the final DataFrame has the correct distribution.

In [ ]:
n_notna = df_combined['name'].notna().sum()
n_na = df_combined['name'].isna().sum()
valid_names = {'kumar', 'parsa', 'yaseen', 'tan', 'daniel'}

assert set(full_cols) == set(df_final.columns).difference({'_from_na'}), (
    'columns not preserved'
)

assert df_final['name'].notna().all(), 'Some rows have NA name'

assert df_final['name'].isin(valid_names).all(), (
    f'Unexpected names: {df_final["name"].unique()}'
)

expected_len = n_notna + n_notna + 2 * n_na
assert len(df_final) == expected_len, (
    f'Expected {expected_len} rows, got {len(df_final)}'
)

old_with_kv = df_old_rows[df_old_rows['key'].notna()]
assert len(old_with_kv) == len(
    df_combined[df_combined['name'].notna() & df_combined['key'].notna()]
), 'Old rows do not have key/value populated (where original had them)'

new_rows = df_final.iloc[n_notna:]
assert new_rows['key'].isna().all(), 'New rows should have NA key'
assert new_rows['value'].isna().all(), 'New rows should have NA value'

grouped = df_final.groupby('instance_id')['name']
assert (grouped.nunique() >= 2).all(), (
    'Some instance_ids have fewer than 2 distinct names: '
    f'{grouped.nunique()[grouped.nunique() < 2]}'
)

new_rows = df_final.iloc[len(df_old_rows) :]
notna_new = new_rows.iloc[: len(notna_idx)]
assert (notna_new['name'] == 'kumar').sum() == len(notna_kumar)
assert (notna_new['name'] == 'parsa').sum() == len(notna_parsa)

paired = df_final[df_final['_from_na']]
kumar_keys = set(
    paired[paired['name'] == 'kumar'][
        ['instance_id', 'image_assets']
    ].itertuples(index=False, name=None)
)
yaseen_keys = set(
    paired[paired['name'] == 'yaseen'][
        ['instance_id', 'image_assets']
    ].itertuples(index=False, name=None)
)
assert kumar_keys == yaseen_keys, (
    f'Mismatch: kumar-only: {kumar_keys - yaseen_keys}, '
    f'yaseen-only: {yaseen_keys - kumar_keys}'
)
parsa_keys = set(
    paired[paired['name'] == 'parsa'][
        ['instance_id', 'image_assets']
    ].itertuples(index=False, name=None)
)
tan_daniel_keys = set(
    paired[paired['name'].isin(['tan', 'daniel'])][
        ['instance_id', 'image_assets']
    ].itertuples(index=False, name=None)
)
assert parsa_keys == tan_daniel_keys, (
    f'Mismatch: parsa-only: {parsa_keys - tan_daniel_keys}, '
    f'tan/daniel-only: {tan_daniel_keys - parsa_keys}'
)

In [ ]:
df_final.to_csv(CSV_BASE / 'running_log.csv')